# 示例: 5. 环状河网模拟示例

**脚本:** `examples/run_looped_network_example.py`

## 目的
演示模型处理带有环路（loops）的复杂河网的能力。
- **核心逻辑:** 对于环状河网，无法通过简单的拓扑排序进行演算。模型为此实现了一个`run_iterative_simulation`方法。在每个时间步内，该方法会反复对整个网络进行迭代计算，直到每个节点的流量都收敛到一个稳定的解。
- **功能:**
    1.  构建一个包含分叉和合流的“菱形”环状河网。
    2.  在网络上游端点输入一个洪水波。
    3.  运行迭代求解器进行模拟。
- **验证:** 最终的图表会展示上游的总入流、环路中两条分支的流量（展示流量如何根据河道属性进行分配），以及在下游汇合后的最终出流过程线。这验证了迭代求解器能够成功处理复杂网络拓扑。

## 1. 环境设置

首先，我们导入所需的库，并把项目的根目录添加到Python的搜索路径中。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sys
import os

# 将项目根目录添加到Python路径
sys.path.insert(0, os.path.abspath(os.path.join(os.path.dirname('__file__'), '..')))
from hydro_model.catchment import Catchment
from hydro_model.routing import MuskingumCungeRouting

## 2. 构建环状河网

我们使用`Catchment`类来构建一个菱形的环状河网。这个网络包含一个分叉点(`N2`)和一合流点(`N5`)。
我们为环路中的两条分支（`R2`和`R3`）设置了不同的物理参数，以观察流量如何根据河道条件进行分配。

In [ ]:
network = Catchment()

nodes = [f"N{i}" for i in range(1, 7)]
for node_id in nodes:
    network.add_node(node_id)

reach_params = {
    "R1": {'length': 2000, 'slope': 0.0005, 'manning_n': 0.04, 'width': 30.0},
    "R2": {'length': 4000, 'slope': 0.0003, 'manning_n': 0.03, 'width': 20.0}, # 西分支
    "R3": {'length': 4000, 'slope': 0.0003, 'manning_n': 0.05, 'width': 25.0}, # 东分支 (更粗糙)
    "R4": {'length': 1000, 'slope': 0.0003, 'manning_n': 0.03, 'width': 20.0},
    "R5": {'length': 1000, 'slope': 0.0003, 'manning_n': 0.05, 'width': 25.0},
    "R6": {'length': 3000, 'slope': 0.0002, 'manning_n': 0.03, 'width': 40.0}
}

network.add_reach("R1", "N1", "N2", MuskingumCungeRouting(**reach_params["R1"]))
network.add_reach("R2", "N2", "N3", MuskingumCungeRouting(**reach_params["R2"]))
network.add_reach("R3", "N2", "N4", MuskingumCungeRouting(**reach_params["R3"]))
network.add_reach("R4", "N3", "N5", MuskingumCungeRouting(**reach_params["R4"]))
network.add_reach("R5", "N4", "N5", MuskingumCungeRouting(**reach_params["R5"]))
network.add_reach("R6", "N5", "N6", MuskingumCungeRouting(**reach_params["R6"]))

print("菱形环状河网构建完成。")

## 3. 定义入流

我们在最上游的节点`N1`处定义一个入流过程线，模拟一个洪水波。其他地方没有侧向入流。

In [ ]:
T = 50
inflow_hydrograph = np.array([0,0,0,10,25,50,70,60,45,30,20,15,10,5,2,1] + [0]*34)
headwater_inflows = {"N1": inflow_hydrograph}
lateral_inflows = {}

## 4. 运行迭代模拟

我们调用`run_iterative_simulation`方法。这个方法会在每个时间步内部进行多次迭代计算，直到环路中每个节点的流量都达到收敛，从而得到一个物理上一致的解。

In [ ]:
print("运行环状河网的迭代模拟...")
reach_q, node_q = network.run_iterative_simulation(headwater_inflows, lateral_inflows, T)
print("模拟完成。")

## 5. 可视化结果

我们将上游总入流、环路中两条分支的流量，以及下游总出流绘制在同一张图上。
从图中可以清晰地看到：
- 在分叉点`N2`，流量被分配到西分支(`R2`)和东分支(`R3`)。
- 由于东分支(`R3`)更粗糙，它的流量小于西分支(`R2`)。
- 在合流点`N5`，两条分支的流量重新汇合，其总和（加上演算效应）等于下游的总出流。

In [ ]:
plt.figure(figsize=(15, 8))
timesteps = np.arange(T)

plt.plot(timesteps, headwater_inflows['N1'], 'k--', label='Inflow at N1')
plt.plot(timesteps, reach_q['R2'], 'b-', label='Flow in West Branch (R2)')
plt.plot(timesteps, reach_q['R3'], 'g-', label='Flow in East Branch (R3)')
plt.plot(timesteps, node_q['N6'], 'r-', linewidth=2, label='Outflow at N6 (Final Outlet)')

plt.title('Simulation of a Looped River Network')
plt.xlabel('Time Step (days)')
plt.ylabel('Flow (m³/s)')
plt.legend()
plt.grid(True)

output_dir = '../examples/results/'
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

output_path = os.path.join(output_dir, 'looped_network_plot.png')
plt.savefig(output_path)
print(f"\n环状河网模拟图已保存到 {output_path}")
plt.show()